<a href="https://colab.research.google.com/github/SagarSawlani/RAG_Article/blob/main/CRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Installations
!pip install langchain langchain-community langchain-huggingface langchain-chroma langchain-groq transformers sentence-transformers torch rank-bm25 chromadb pymupdf pypdf python-dotenv tqdm numpy pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/6

In [25]:
import os
import json
from dotenv import load_dotenv

# Document Loading
from langchain_community.document_loaders import PyMuPDFLoader

# Text Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Database
from langchain_chroma import Chroma

# LLM
from langchain_groq import ChatGroq

# Prompt Template
from langchain_core.prompts import ChatPromptTemplate

# BM25
from langchain_community.retrievers import BM25Retriever

# Hybrid Retrieval
from langchain_classic.retrievers import EnsembleRetriever

# Cross Encoder
from sentence_transformers import CrossEncoder

from tqdm import tqdm

In [3]:
load_dotenv("/content/drive/MyDrive/RAG_Article/.env")
GROQ_API_KEY = os.environ['GROQ_API_KEY']
if GROQ_API_KEY is None:
  raise ValueError("GROQ_API_KEY is not set")

In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
DATA_PATH = "/content/drive/MyDrive/RAG_Article/data"

In [6]:
documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyMuPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages.")

Loaded 36 pages.


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 30


In [8]:
# ChromaDB Path
CHROMA_DB_DIR = "/content/drive/MyDrive/RAG_Article/chroma_db"

In [9]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=CHROMA_DB_DIR
)

In [10]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)

In [11]:
bm25_retriever = BM25Retriever.from_documents(chunks)

In [12]:
bm25_retriever.k = 4

In [13]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[
        dense_retriever,
        bm25_retriever
    ],
    weights=[0.5, 0.5]
)

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer ONLY using the provided context.

If the answer is not contained in the context, say:
"I couldn't find that information in the provided documents."

Context:
{context}

Question:
{question}

Answer:
""")

In [15]:
reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [16]:
def rerank_documents(query, documents, top_k=5):
    # Create (query, document) pairs
    sentence_pairs = [
        (query, doc.page_content)
        for doc in documents
    ]

    # Predict relevance scores
    scores = reranker.predict(sentence_pairs)

    # Combine documents with scores
    ranked_docs = sorted(
        zip(documents, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Return top_k
    return ranked_docs[:top_k]

In [17]:
query = "What are the different array operations?"

candidate_docs = ensemble_retriever.invoke(query)

In [18]:
reranked_docs = rerank_documents(
    query,
    candidate_docs,
    top_k=2
)

final_docs = [doc for doc, score in reranked_docs]

In [22]:
evaluation_prompt = ChatPromptTemplate.from_template("""
You are an expert Retrieval Quality Evaluator.

Given:

1. User Question
2. Retrieved Chunks

Evaluate whether the retrieved chunks are sufficient to answer the question.

Return ONLY valid JSON.

Schema:

{{
    "action":"ACCEPT" or "RETRIEVE_AGAIN",

    "overall_confidence":0.0,

    "reason":"brief reason",

    "chunks":[
        {{
            "chunk_id":0,
            "relevance":0.95,
            "keep":true,
            "reason":"Directly answers the question."
        }}
    ]
}}

Rules:

- overall_confidence must be between 0 and 1.
- keep=true only if the chunk is useful.
- If most chunks are irrelevant, choose RETRIEVE_AGAIN.
- Return ONLY JSON.

Question:

{question}

Retrieved Chunks:

{context}
""")

In [23]:
def evaluate_retrieval(question, docs):

    context = ""

    for i, doc in enumerate(docs):
        context += f"""
Chunk {i}

{doc.page_content}

---------------------
"""

    prompt = evaluation_prompt.format(
        question=question,
        context=context
    )

    response = llm.invoke(prompt)

    return response.content

In [27]:
def parse_evaluation(result):

    result = result.strip()

    if result.startswith("```"):
        result = result.replace("```json","")
        result = result.replace("```","")

    return json.loads(result)

In [28]:
evaluation = evaluate_retrieval(query, final_docs)

evaluation_json = parse_evaluation(evaluation)

filtered_docs = []

for chunk in evaluation_json["chunks"]:

    if chunk["keep"]:

        filtered_docs.append(
            final_docs[chunk["chunk_id"]]
        )

print(len(filtered_docs))

1


In [29]:
rewrite_prompt = ChatPromptTemplate.from_template("""
You are an expert search query optimizer.

Rewrite the user's question so that it retrieves better and relevant information .

Rules:

- Preserve intent.
- Add technical terminology if appropriate.
- Make the query specific.
- Return ONLY the rewritten query.

Question:

{question}
""")

In [30]:
action = evaluation_json["action"]

print(action)

ACCEPT


In [39]:
def rewrite_query(question):

    prompt = rewrite_prompt.format(
        question=question
    )

    response = llm.invoke(prompt)

    return response.content.strip()

In [31]:
if action == "RETRIEVE_AGAIN":

    new_query = rewrite_query(query)

    print("Original:", query)
    print("Rewritten:", new_query)

    candidate_docs = ensemble_retriever.invoke(new_query)

    reranked_docs = rerank_documents(
        new_query,
        candidate_docs,
        top_k=5
    )

    new_docs = [
        doc
        for doc, score in reranked_docs
    ]

In [33]:
merged_docs = filtered_docs.copy()

if action == "RETRIEVE_AGAIN":

    existing = {
        doc.page_content
        for doc in merged_docs
    }

    for doc in new_docs:

        if doc.page_content not in existing:

            merged_docs.append(doc)

In [34]:
reranked_docs = rerank_documents(
    query,
    merged_docs,
    top_k=5
)

final_docs = [
    doc
    for doc, score in reranked_docs
]

In [35]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [36]:
context = format_docs(final_docs)

formatted_prompt = prompt.format(
    context=context,
    question=query
)

In [37]:
response = llm.invoke(formatted_prompt)
print(response.content)

The different array operations are:

- Searching an element  
- Inserting a new element  
- Deleting a required element  
- Modifying an element  
- Merging arrays
